<a href="https://colab.research.google.com/github/karthikpaii/workshop-MITE/blob/main/workshopday2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
pip install psycopg


In [49]:
import os
import sys
from typing import List, Literal, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
import psycopg
from psycopg.rows import dict_row

In [50]:
MOST_RUNS_TEST_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=batting&slug=batting_most_runs&format=test"
MOST_RUNS_ODI_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=batting&slug=batting_most_runs&format=odi"
TOP_WICKET_TAKERS_ODI_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=bowling&slug=bowling_top_wicket_takers&format=odi"
TOP_WICKET_TAKERS_TEST_URL = "https://www.bcci.tv/getStats?platform=international&type=men&s_type=bowling&slug=bowling_top_wicket_takers&format=test"

In [51]:
OUT_DIR = os.path.join(os.getcwd(), "out")

In [52]:
Discipline = Literal["batting", "bowling"]

jobs: list[tuple[str, str, Discipline]] = [
    (MOST_RUNS_TEST_URL, "bcci_test_most_runs", "batting"),
    (MOST_RUNS_ODI_URL, "bcci_odi_most_runs", "batting"),
    (TOP_WICKET_TAKERS_TEST_URL, "bcci_test_top_wickets", "bowling"),
    (TOP_WICKET_TAKERS_ODI_URL, "bcci_odi_top_wickets", "bowling"),
]

In [53]:
columns: dict[Discipline, list[str]] = {
"batting": [
        "Rank",
        "Name",
        "Matches",
        "Inns",
        "Avg",
        "SR",
        "HS",
        "Fours",
        "Sixes",
        "Fifties",
        "Centuries",
        "Runs",
    ],
    "bowling": [
        "Rank",
        "Name",
        "Matches",
        "Inns",
        "Avg",
        "Econ",
        "SR",
        "BBI",
        "Four_w",
        "Five_w",
        "Wkts",
    ],
}

In [54]:
saved_paths: List[str] = []

In [55]:
os.makedirs(OUT_DIR,exist_ok=True)

In [61]:
html_score:dict[str,str]=[]

In [72]:
import os
import requests


os.makedirs(OUT_DIR, exist_ok=True)

html_store: dict[str, str] = {}

for url, basename, kind in jobs:

        resp = requests.get(url, timeout=30)
        resp.raise_for_status()

        payload = resp.json()
        html = payload.get("html")

        if not html:
            print(f"HTML data found for {basename} does not exist.")
            continue


        html_store[basename] = html


        print(f"Got HTML data for {basename}")




Got HTML data for bcci_test_most_runs
Got HTML data for bcci_odi_most_runs
Got HTML data for bcci_test_top_wickets
Got HTML data for bcci_odi_top_wickets


In [73]:
def normalize_label(text: str) -> str:
   low=text.replace("\u2019","").replace()
   mapping = {
            "matches": "Matches",
            "inns": "Inns",
            "avg": "Avg",
            "sr": "SR",
            "hs": "HS",
            "runs": "Runs",
            "4's": "Fours",
            "4s": "Fours",
            "6's": "Sixes",
            "6s": "Sixes",
            "50's": "Fifties",
            "50s": "Fifties",
            "100's": "Centuries",
            "100s": "Centuries",
            "econ": "Econ",
            "economy": "Econ",
            "wkts": "Wkts",
            "wickets": "Wkts",
            "bbi": "BBI",
            "best bowling": "BBI",
            "best": "BBI",
            "4w": "Four_w",
            "5w": "Five_w",
        }


   return mapping.get(low,low)

def coerce_value(text: str):
  try:
    if "." in text:
      return float(text)
    return int(text)
  except Exception:
    return text.strip()



In [74]:
def extract_data_from_html(html: str, kind: Discipline) -> tuple[list[dict], list[str]]:
    """Parse rows from the getStats API HTML snippet."""

    # TODO: parse the HTML string using BeautifulSoup and the "lxml" parser
    # soup = BeautifulSoup(html, "lxml")

    # TODO: find the statistics table element (use the correct CSS selector)
    # table = soup.select_one(".stats-data-table-player table")
    # if table is None:
    #     return []

    records: list[dict] = []

    # TODO: get the first ranking player (separate section in the HTML) and append
    # records.append(get_first_rank_player(soup))

    # Process each row in the table
    for tr in table.select("tr"):
        tds = tr.find_all("td")
        if len(tds) < 3:  # Skip rows with insufficient data
            continue

        # Extract rank (serial number) from first column
        sn_el = tds[0].find(["h5", "h6"]) or tds[0]
        name_el = tds[1].find("h6") or tds[1]
        try:
            sn = int((sn_el.get_text(strip=True) or "0").strip())
        except Exception:
            sn = None

        # Extract player name from second column
        name = name_el.get_text(strip=True)
        row: dict = {"Rank": sn, "Name": name}

        # Extract statistics from remaining columns (matches, innings, avg, etc.)
        for td in tds[2:]:
            val_el = td.find("h6")  # Value element
            lab_el = td.find("span")  # Label element
            if not val_el or not lab_el:
                continue

            val_txt = val_el.get_text(strip=True)
            lab_txt = lab_el.get_text(strip=True)
            key = normalize_label(lab_txt)  # Normalize label (e.g., "4's" -> "Fours")
            row[key] = coerce_value(val_txt)  # Convert value to appropriate type

        records.append(row)

    # Ensure all records have all columns (fill missing with None)
    for r in records:
        for c in columns[kind]:
            if c not in r:
                r[c] = None

    return records